#### [HMDA dataset](https://ffiec.cfpb.gov/data-browser/data/2022?category=counties&items=47037)

In [51]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import seaborn as sns

### Read in the data

In [3]:
hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv') #.dropna().reset_index(drop = True)

C:\Users\Cat\AppData\Local\Temp\ipykernel_28476\2529231324.py:4: DtypeWarning: Columns (22,23,24,26,27,28,29,30,31,32,33,38,43,44) have mixed types. Specify dtype option on import or set low_memory=False.
  hmda_2022 = pd.read_csv('../data/msamd_12060_2022.csv') #.dropna().reset_index(drop = True)


In [50]:
hmda_2022.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2022,549300FGXN1K3HLB1R50,12060,GA,13135.0,1.313505e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Ethnicity Not Available,...,NaN,NaN,NaN,4582,90.27,95700,59.25,693,1364,26
1,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,5777,67.49,95700,117.88,1806,2070,29


In [7]:
len(hmda_2022.columns)

99

In [52]:
# https://stackoverflow.com/questions/15943769/how-do-i-get-the-row-count-of-a-pandas-dataframe
# (== number of non-NaN values in first column)
hmda_2022[hmda_2022.columns[0]].count()

np.int64(378284)

### Remove all values that do not reflect approved, denied, or approved but not accepted

In [9]:
# remove all values that do not reflect approved, denied, or approved but not accepted
# Description: The action taken on the covered loan or application
# Values:
# 1 - Loan originated
# 2 - Application approved but not accepted
# 3 - Application denied
# 4 - Application withdrawn by applicant
# 5 - File closed for incompleteness
# 6 - Purchased loan
# 7 - Preapproval request denied
# 8 - Preapproval request approved but not accepted

list_of_values_to_keep = [1,2,3]

hmda_2022_action_taken = hmda_2022[hmda_2022['action_taken'].isin(list_of_values_to_keep)]
hmda_2022_action_taken['action_taken'].unique()

array([1, 2, 3])

In [53]:
# (== number of non-NaN values in first column)
hmda_2022_action_taken[hmda_2022_action_taken.columns[0]].count()

np.int64(254012)

### Remove not applicable (could include applicants with little to no credit) and exempt institutions to eliminate legal entitities like corporations etc from being included in the data instead of individuals

In [11]:
# remove not applicable (could include applicants with little to no credit) and exempt institutions
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8]

hmda_2022_credit_score_app = hmda_2022_action_taken[hmda_2022_action_taken['applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_2022_credit_score_app['applicant_credit_score_type'].unique()

array([2, 1, 3, 8, 7, 6, 4, 5])

In [12]:
# (== number of non-NaN values in first column)
hmda_2022_credit_score_app[hmda_2022_credit_score_app.columns[0]].count()

np.int64(222254)

In [13]:
# remove not applicable (could include applicants with little to no credit) and exempt institutions
# to eliminate legal entitities like corporations etc from being included in the data instead of individuals

# Description: The name and version of the credit scoring model used to generate the credit score, or scores, relied on in making the credit decision
# Values:
# 1 - Equifax Beacon 5.0
# 2 - Experian Fair Isaac
# 3 - FICO Risk Score Classic 04
# 4 - FICO Risk Score Classic 98
# 5 - VantageScore 2.0
# 6 - VantageScore 3.0
# 7 - More than one credit scoring model
# 8 - Other credit scoring model
# 9 - Not applicable
# 10 - No co-applicant
# 11 – FICO Score 9
# 12 – FICO Score 8
# 13 – FICO Score 10
# 14 – FICO Score 10T
# 15 - VantageScore 4.0
# 1111 - Exempt

list_of_values_to_keep = [1,2,3,4,5,6,7,8,10,11,12,13,14,15]

hmda_2022_credit_score_co = hmda_2022_credit_score_app[hmda_2022_credit_score_app['co-applicant_credit_score_type'].isin(list_of_values_to_keep)]
hmda_2022_credit_score_co['co-applicant_credit_score_type'].unique()

array([10,  2,  3,  1,  8,  6,  7,  4,  5, 11])

In [57]:
# (== number of non-NaN values in first column)
hmda_2022_credit_score_co[hmda_2022_credit_score_co.columns[0]].count()

np.int64(181011)

### Inspect all columns

In [82]:
hmda_2022_credit_score_co.columns

Index(['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code',
       'census_tract', 'conforming_loan_limit', 'derived_loan_product_type',
       'derived_dwelling_category', 'derived_ethnicity', 'derived_race',
       'derived_sex', 'action_taken', 'purchaser_type', 'preapproval',
       'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage',
       'open-end_line_of_credit', 'business_or_commercial_purpose',
       'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread',
       'hoepa_status', 'total_loan_costs', 'total_points_and_fees',
       'origination_charges', 'discount_points', 'lender_credits', 'loan_term',
       'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization',
       'interest_only_payment', 'balloon_payment',
       'other_nonamortizing_features', 'property_value', 'construction_method',
       'occupancy_type', 'manufactured_home_secured_property_type',
       'manufactured_home_land_property_interest', 'total_

### Create a unique index for each application

In [67]:
# create a unique number for every application as an ID column of sorts
hmda_2022_credit_score_co["unique_id"] = range(len(hmda_2022_credit_score_co))

C:\Users\Cat\AppData\Local\Temp\ipykernel_28476\2598958234.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hmda_2022_credit_score_co["unique_id"] = range(len(hmda_2022_credit_score_co))


### Melt the AUS columns into a smaller df to OHE those columns

In [68]:
# melting data into small df to ohe it
hmda_melted = hmda_2022_credit_score_co.melt(
    id_vars="unique_id", 
    value_vars=["aus-1", "aus-2", "aus-3", "aus-4", "aus-5"],
    var_name="aus_types", 
    value_name="aus_values"
) 

In [69]:
hmda_melted.head(2)

,unique_id,aus_types,aus_values
0,0,aus-1,1.0
1,1,aus-1,1.0


In [70]:
hmda_melted["aus_values"].unique()

array([1.000e+00, 6.000e+00, 2.000e+00, 3.000e+00, 5.000e+00, 4.000e+00,
       1.111e+03, 7.000e+00,       nan])

### Fill in NaNs with 0s and change it to an integer not a float

In [71]:
hmda_melted["aus_values"] = hmda_melted["aus_values"].fillna(0).astype(int)

In [78]:
hmda_melted.head(2)

,unique_id,aus_types,aus_values
0,0,aus-1,1
1,1,aus-1,1


### OneHotEncode 

In [73]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_melted[["aus_values"]])

# combine the encoded columns back with the original df
hmda_ohe = pd.concat([hmda_melted.drop(columns=["aus_values"]), encoded_df], axis=1)

In [74]:
hmda_ohe.head(2)#["aus_values_7.0"].unique()#.head(2)

,unique_id,aus_types,aus_values_0,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,0,aus-1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,aus-1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [79]:
hmda_ohe.columns.unique()

Index(['unique_id', 'aus_values_1', 'aus_values_2', 'aus_values_3',
       'aus_values_4', 'aus_values_5', 'aus_values_6', 'aus_values_7',
       'aus_values_1111'],
      dtype='object')

#### Drop unnecessary columns 

In [80]:
hmda_ohe = hmda_ohe.drop(columns=["aus_values_0", "aus_types"])

KeyError: "['aus_values_0', 'aus_types'] not found in axis"

### Group by the "unique_id" column, add those values together, and reset the index

In [77]:
auses = hmda_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [39]:
auses["aus_values_4"].unique()

array([0., 1., 4., 5.])

In [43]:
auses.head(2)

,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [81]:
hmda_2022_credit_score_co.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,unique_id
1,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5777,67.49,95700,117.88,1806,2070,29,0
2,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5502,22.41,95700,141.54,1798,2180,25,1


### Merge the AUS df and the original HMDA df together

In [45]:
hmda_aus = hmda_2022_credit_score_co.merge(auses, on="unique_id")

In [47]:
hmda_aus.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,tract_median_age_of_housing_units,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,29,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,25,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Melt the applicant_ethnicity columns into a smaller df to OHE those columns

In [83]:
# melting data into small df to ohe it
hmda_melted = hmda_2022_credit_score_co.melt(
    id_vars="unique_id", 
    value_vars=["aus-1", "aus-2", "aus-3", "aus-4", "aus-5"],
    var_name="aus_types", 
    value_name="aus_values"
) 

In [ ]:
hmda_melted.head(2)

In [ ]:
hmda_melted["aus_values"].unique()

### Fill in NaNs with 0s and change it to an integer not a float

In [ ]:
hmda_melted["aus_values"] = hmda_melted["aus_values"].fillna(0).astype(int)

In [ ]:
hmda_melted.head(2)

### OneHotEncode 

In [ ]:
# use OneHotEncoder 
transformed = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

# fit and transform the specific column
encoded_df = transformed.fit_transform(hmda_melted[["aus_values"]])

# combine the encoded columns back with the original df
hmda_ohe = pd.concat([hmda_melted.drop(columns=["aus_values"]), encoded_df], axis=1)

In [ ]:
hmda_ohe.head(2)#["aus_values_7.0"].unique()#.head(2)

In [84]:
hmda_ohe.columns.unique()

Index(['unique_id', 'aus_values_1', 'aus_values_2', 'aus_values_3',
       'aus_values_4', 'aus_values_5', 'aus_values_6', 'aus_values_7',
       'aus_values_1111'],
      dtype='object')

#### Drop unnecessary columns 

In [ ]:
hmda_ohe = hmda_ohe.drop(columns=["aus_values_0", "aus_types"])

### Group by the "unique_id" column, add those values together, and reset the index

In [ ]:
auses = hmda_ohe.groupby("unique_id").sum().reset_index()#.astype(int)

In [ ]:
auses["aus_values_4"].unique()

In [85]:
auses.head(2)

,unique_id,aus_values_1,aus_values_2,aus_values_3,aus_values_4,aus_values_5,aus_values_6,aus_values_7,aus_values_1111
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [86]:
hmda_2022_credit_score_co.head(2)

,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units,unique_id
1,2022,549300FGXN1K3HLB1R50,12060,GA,13113.0,1.311314e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5777,67.49,95700,117.88,1806,2070,29,0
2,2022,549300FGXN1K3HLB1R50,12060,GA,13077.0,1.307717e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,5502,22.41,95700,141.54,1798,2180,25,1


### Merge the AUS df and the original HMDA df together

In [ ]:
hmda_aus = hmda_2022_credit_score_co.merge(auses, on="unique_id")

In [ ]:
hmda_aus.head(2)

### Melt the co-applicant_ethnicity columns into a smaller df to OHE those columns

### Melt the applicant_race columns into a smaller df to OHE those columns

### Melt the co-applicant_race columns into a smaller df to OHE those columns

### Melt the denial_reason columns into a smaller df to OHE those columns


In [32]:
# # remove not applicable (could include applicants with little to no credit) and exempt institutions
# # retain only records named federal AUS

# # aus-1
# # Description: The automated underwriting system(s) (AUS) used by the financial institution to evaluate the application
# # Values:
# # 1 - Desktop Underwriter (DU)
# # 2 - Loan Prospector (LP) or Loan Product Advisor
# # 3 - Technology Open to Approved Lenders (TOTAL) Scorecard
# # 4 - Guaranteed Underwriting System (GUS)
# # 5 - Other
# # 6 - Not applicable
# # 7 - Internal Proprietary System
# # 1111 - Exempt

# list_of_values_to_keep = [1,2,3,4]

# hmda_2022_credit_score_aus1 = hmda_2022_credit_score_co[hmda_2022_credit_score_co['aus-1'].isin(list_of_values_to_keep)]
# hmda_2022_credit_score_aus1['aus-1'].unique()